# Function 8 Analysis - Week 3

This notebook contains the exploratory analysis for **Function 8** in Week 3. After incorporating the latest two measurements we now have **42 datapoints** informing the Bayesian optimiser.

**Function Description:** You're optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown. Your objective is to find the parameter combination that maximises the function's output, such as performance, efficiency or validation accuracy. Because the function is high-dimensional and likely complex, global optimisation is hard, so identifying strong local maxima is often a practical strategy.


## Loading and Displaying the Data

We load the inputs and outputs for function 8 and display them in a table format to inspect the raw data values. Week 1's point yielded ≈9.74, and Week 2's point `(0.1, 0.1, 0.1, 0.1, 0.1, 0.4, 0.25, 0.45)` achieved ≈9.73, which is our **second highest point**. Interestingly, Week 2's configuration has very different values for x1-x6 compared to Week 1, but similar x7 and x8 values, suggesting that **x7 and x8 are the key features** while the others matter less.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_8")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Add the new points from Week 1 and Week 2
X_new_point_week_1 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.250000, 0.450000]])
y_new_point_week_1 = np.array([9.74365])
X_new_point_week_2 = np.array([[0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.400000, 0.250000, 0.450000]])
y_new_point_week_2 = np.array([9.73005])

X = np.vstack([X, X_new_point_week_1, X_new_point_week_2])
y = np.concatenate([y, y_new_point_week_1, y_new_point_week_2])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


df sorted by y


,x1,x2,x3,x4,x5,x6,x7,x8,y,x_avg
0,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.250000,0.450000,9.743650,0.050000
1,0.100000,0.100000,0.100000,0.100000,0.100000,0.400000,0.250000,0.450000,9.730050,0.100000
2,0.056447,0.065956,0.022929,0.038786,0.403935,0.801055,0.488307,0.893085,9.598482,0.046030
3,0.192640,0.630677,0.416796,0.490529,0.796086,0.654567,0.276241,0.295518,9.344274,0.432661
4,0.481245,0.102461,0.219486,0.677322,0.247509,0.244341,0.163825,0.715962,9.183005,0.370129
5,0.145120,0.119328,0.420888,0.387609,0.155423,0.875172,0.510560,0.728611,9.141639,0.268236
6,0.044329,0.013581,0.258198,0.577644,0.051280,0.158563,0.591030,0.077953,9.013075,0.223438
7,0.143550,0.937415,0.232325,0.009043,0.414579,0.409325,0.553779,0.205841,8.976554,0.330583
8,0.028947,0.028279,0.481372,0.613175,0.672660,0.022113,0.601483,0.524885,8.830745,0.287943
9,0.338954,0.566932,0.376751,0.098916,0.659452,0.245548,0.762483,0.732153,8.817558,0.345388


## Bayesian Optimization Setup

We use Gaussian Process (GP) regression to model the unknown function based on our observed data. The GP provides both a mean prediction and uncertainty estimates. The search space is defined as [0, 1] for each of the eight input variables.

**Strategy Evolution:**
- **Week 1:** Used UCB to find a point with low x1-x4 and moderate x5-x8, which scored ≈9.74.
- **Week 2:** Continued with UCB, which found a point with very different x1-x6 values but similar x7 (0.25) and x8 (0.45), scoring ≈9.73 (second highest). This suggests x7 and x8 are the key features.
- **Week 3:** Given that x7 and x8 appear to be the critical features, we implement an **exploitation strategy** that:
  - Keeps x1-x6 fixed at the best point's values
  - Focuses optimization on x7 and x8 only
  - Uses Expected Improvement (EI) with low exploration to exploit the promising x7-x8 region


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
np.random.seed(42)
# Per-feature lengthscales with bounds (assuming little noise, no WhiteKernel)
kernel = Matern(
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.2, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.length_scale)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [1.18589116 1.79603299 0.86282553 2.34507505 4.52728751 5.
 1.21751408 5.        ]
Full kernel parameters: {'length_scale': array([1.18589116, 1.79603299, 0.86282553, 2.34507505, 4.52728751,
       5.        , 1.21751408, 5.        ]), 'length_scale_bounds': (0.2, 5.0), 'nu': 2.5}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## Finding the Next Point to Evaluate (Week 3 - Exploitation Strategy)

Based on Week 2's result showing that x7 and x8 are the critical features (while x1-x6 can vary widely), we implement an **exploitation strategy**:
- **Fix x1-x6** at the best point's values
- **Optimize only x7 and x8** using Expected Improvement (EI) with low exploration (`xi=0.001`)
- This focuses our search on the promising x7-x8 region while keeping other features at proven good values


In [3]:
from scipy.stats import norm

# Expected Improvement acquisition function
def expected_improvement(x, gp, y_best, xi=0.001):
    """
    Expected Improvement acquisition function.
    
    Args:
        x: Point to evaluate (full 8D point)
        gp: Fitted Gaussian Process
        y_best: Best observed value so far
        xi: Exploration-exploitation trade-off parameter (small values favor exploitation)
    
    Returns:
        Negative EI (for minimization)
    """
    x = x.reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    
    # Add small epsilon to avoid division by zero
    sigma = sigma + 1e-9
    
    # Calculate improvement
    improvement = mu - y_best - xi
    Z = improvement / sigma
    
    # Expected Improvement formula
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    
    return -ei[0]  # Return negative for minimization

# Get current best point
y_best = y.max()
best_idx = y.argmax()
best_point = X[best_idx].copy()

print(f"Current best score: {y_best:.4f}")
print(f"Current best point: {best_point}")
print(f"\nExploitation strategy: Fixing x1-x6 at best point values, optimizing x7 and x8 only")
print(f"  Fixed values: x1={best_point[0]:.4f}, x2={best_point[1]:.4f}, x3={best_point[2]:.4f}, x4={best_point[3]:.4f}, x5={best_point[4]:.4f}, x6={best_point[5]:.4f}")

# Wrapper function that fixes x1-x6 and only optimizes x7-x8
def acquisition_x7x8(x7x8):
    """
    Acquisition function that only optimizes x7 and x8, keeping x1-x6 fixed.
    """
    x_full = best_point.copy()
    x_full[6] = x7x8[0]  # x7
    x_full[7] = x7x8[1]  # x8
    return expected_improvement(x_full, gp, y_best, xi=0.001)

# Optimize only x7 and x8 with multiple random restarts
bounds_x7x8 = [(0, 1), (0, 1)]
n_restarts = 20
best_acquisition = np.inf
best_x7x8 = None

np.random.seed(42)
for i in range(n_restarts):
    # Start near the best point's x7, x8 values
    if i < 10:
        x0 = best_point[6:8].copy()
        x0 += np.random.normal(0, 0.1, 2)
        x0 = np.clip(x0, 0, 1)
    else:
        x0 = np.random.uniform(0, 1, 2)
    
    result = minimize(
        acquisition_x7x8,
        x0=x0,
        bounds=bounds_x7x8,
        method='L-BFGS-B'
    )
    
    if result.fun < best_acquisition:
        best_acquisition = result.fun
        best_x7x8 = result.x

# Construct the full recommended point
next_point = best_point.copy()
next_point[6] = best_x7x8[0]  # x7
next_point[7] = best_x7x8[1]  # x8

mu_pred, sigma_pred = gp.predict(next_point.reshape(1, -1), return_std=True)

print(f"\n{'='*60}")
print("BAYESIAN OPTIMIZATION RECOMMENDATION (Exploitation Strategy)")
print(f"{'='*60}")
print(f"\nNext point to evaluate:")
print(f"  x1={next_point[0]:.4f}, x2={next_point[1]:.4f}, x3={next_point[2]:.4f}, x4={next_point[3]:.4f}")
print(f"  x5={next_point[4]:.4f}, x6={next_point[5]:.4f}, x7={next_point[6]:.4f}, x8={next_point[7]:.4f}")
print(f"\nPredicted output: {mu_pred[0]:.4f} ± {sigma_pred[0]:.4f}")
print(f"Expected Improvement: {-best_acquisition:.6f}")
print(f"\nNote: x1-x6 are fixed at best point values, only x7 and x8 were optimized.")


Current best score: 9.7437
Current best point: [0.05 0.05 0.05 0.05 0.6  0.93 0.25 0.45]

Exploitation strategy: Fixing x1-x6 at best point values, optimizing x7 and x8 only
  Fixed values: x1=0.0500, x2=0.0500, x3=0.0500, x4=0.0500, x5=0.6000, x6=0.9300

BAYESIAN OPTIMIZATION RECOMMENDATION (Exploitation Strategy)

Next point to evaluate:
  x1=0.0500, x2=0.0500, x3=0.0500, x4=0.0500
  x5=0.6000, x6=0.9300, x7=0.0000, x8=1.0000

Predicted output: 9.6724 ± 0.2078
Expected Improvement: 0.051727

Note: x1-x6 are fixed at best point values, only x7 and x8 were optimized.


## Distance Analysis of Recommended Point

We calculate the Euclidean distance from the recommended point to all existing observations. This helps us understand how similar the recommended point is to our existing data. We also compute the average y value of the three closest neighbors to get an estimate of the expected output at the recommended point.


In [4]:
distances = np.sqrt(((X - next_point)**2).sum(axis=1))
df_dist = pd.DataFrame({"point_index": range(len(X)), "distance": distances, "y": y})
df_dist = df_dist.sort_values("distance")
print("Euclidean distances from recommended point to all observations:")
print(df_dist.to_string(index=False))
closest_3 = df_dist.head(3)
avg_y = closest_3["y"].mean()
print(f"\nThree closest neighbors: points {closest_3['point_index'].tolist()}")
print(f"Average y value of closest 3 neighbors: {avg_y:.4f}")


Euclidean distances from recommended point to all observations:
 point_index  distance        y
          14  0.553261 9.598482
          40  0.604152 9.743650
          22  0.894634 9.141639
          41  0.951788 9.730050
          32  1.103106 8.278062
          39  1.145809 9.183005
          26  1.171774 9.344274
          31  1.176114 8.421759
           5  1.208037 8.541748
          30  1.231478 7.923759
          23  1.258969 8.817558
           0  1.283867 7.398721
           4  1.374960 8.606117
          25  1.386079 8.830745
           6  1.431250 7.327435
          12  1.440145 8.976554
          10  1.453088 7.854541
          35  1.454306 8.472936
          36  1.460145 7.977685
          13  1.462371 7.379083
          15  1.470996 8.159983
          18  1.478772 7.433744
          27  1.495033 6.887846
          38  1.510436 7.436594
           2  1.539486 8.459482
          28  1.542933 8.042213
          19  1.555252 9.013075
           1  1.590568 7.005227
        

## Analysis and Recommendation

This is a pure black box optimisation in high dimensional space. We used a "standard" Bayesian optimisation to model this. The next recommended point is x1=0.0399, x2=0.0000, x3=0.1137, x4=0.0000, x5=1.0000, x6=0.9283, x7=0.0000, x8=0.0000. Checking against our best point, the GP has learned "low x1–x4, high x6" but is suggesting extreme values (0.0, 1.0) for x5, x7, x8, regions with little data and big differences to our best point, suggesting the model is exploring. The predicted y is 9.8015 with a high error of 0.3873. More data seems to be needed to get more confident. We decide to be a bit more conservative, keeping x1–x4 and x6 close the GP, but moderating x5, x7, x8 away from extremes: [0.05, 0.05, 0.05, 0.05, 0.60, 0.93, 0.25, 0.45]